<a href="https://colab.research.google.com/github/steffiprog/ML/blob/main/7_%D0%A1NN/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HabrDifficultyCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes, output_dim, dropout, pad_idx):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim,
                      out_channels=n_filters,
                      kernel_size=fs)
            for fs in filter_sizes
        ])

        self.fc = nn.Linear(len(filter_sizes) * n_filters, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.embedding(text)
        embedded = embedded.permute(0, 2, 1)

        pooled = []
        for conv in self.convs:
            conved = F.relu(conv(embedded))
            max_pooled = F.max_pool1d(conved, conved.shape[2]).squeeze(2)
            pooled.append(max_pooled)

        cat = self.dropout(torch.cat(pooled, dim=1))

        return self.fc(cat)